# Spatial Analysis of Identified Clones

This notebook analyzes the spatial distribution patterns of cell clusters/clones identified in Step 6A.

## Analysis Components

1. **Spatial Metrics** - Compute dispersion, density, and extent of each clone
2. **Randomness Testing** - Test whether clones are spatially clustered, dispersed, or random
3. **Clone Interactions** - Analyze spatial relationships between different clones
4. **Boundary Analysis** - Test for enrichment at tumor periphery vs core
5. **Visualization** - Generate spatial distribution maps

## Prerequisites

- Completed Step 6A (Leiden clustering/subclustering)
- Unified CSV file with cluster assignments
- Spatial coordinates (x_centroid, y_centroid) in the data

In [ ]:
# Imports & Environment Setup
import os, sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

# Add src to path for imports
project_root = Path().resolve().parent
src_path = project_root / "src"
sys.path.insert(0, str(src_path))

from spacemap.config import default_config
from spacemap.spatial_analysis_utils import (
    compute_clone_spatial_metrics,
    test_spatial_randomness,
    compute_clone_interactions,
    compute_distance_from_boundary,
    test_clone_boundary_enrichment,
    plot_clone_spatial_distribution,
    plot_clone_metrics_summary
)

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

print(f"Project root: {project_root}")
print("Spatial Analysis Setup Complete")

## Configuration

Configure input files and analysis parameters.

In [ ]:
# =============================================================================
# CONFIGURATION
# =============================================================================

# I/O Paths
INPUT_DIR = project_root / 'outputs/phenotyping'
OUTPUT_DIR = INPUT_DIR / 'spatial_analysis'
PLOTS_DIR = OUTPUT_DIR / 'plots'
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# Input file (from Step 6A or Step 7)
INPUT_FILE = INPUT_DIR / 'all_samples_combined_corrected_phenotyped.csv'

# Column Configuration
CLONE_COLUMN = 'leiden_cluster_tumor'  # Change to your cluster column name
# CLONE_COLUMN = 'leiden_cluster_mixed'  # Example: for mixed subclustering

# Spatial columns
SPATIAL_COLS = ['x_centroid', 'y_centroid']

# Grouping columns defining independent spatial regions
# These define separate spatial contexts (different patients, tissues, tumor regions)
GROUP_COLS = ['batch_id', 'tissue_roi', 'tumor_roi']

# Analysis Parameters
INTERACTION_RADIUS = 50  # Radius for defining spatial neighbors (adjust based on your pixel scale)
BOUNDARY_THRESHOLD = 100  # Distance threshold for boundary vs core (adjust based on your scale)
N_SIMULATIONS = 1000  # Number of Monte Carlo simulations for randomness test
RANDOM_SEED = 42  # For reproducibility

# Filtering (optional)
# Only analyze specific regions if needed
FILTER_CRITERIA = {
    # 'batch_id': ['P11', 'P15', 'P17'],  # Example: specific patients
    # 'tumor': 1,  # Example: only tumor regions
}

print("Configuration loaded:")
print(f" Input file: {INPUT_FILE}")
print(f" Clone column: {CLONE_COLUMN}")
print(f" Spatial columns: {SPATIAL_COLS}")
print(f" Grouping: {GROUP_COLS}")
print(f" Output directory: {OUTPUT_DIR}")

## Step 1: Load Data

Load the unified phenotyped data with cluster assignments.

In [ ]:
# =============================================================================
# STEP 1: DATA LOADING
# =============================================================================
print("Step 1: Loading data...")

if not INPUT_FILE.exists():
    raise FileNotFoundError(f"Input file not found: {INPUT_FILE}")

data = pd.read_csv(INPUT_FILE, low_memory=False)
print(f"Loaded: {len(data):,} cells")

# Apply filters if specified
if FILTER_CRITERIA:
    for col, val in FILTER_CRITERIA.items():
        if col in data.columns:
            if isinstance(val, list):
                data = data[data[col].isin(val)]
            else:
                data = data[data[col] == val]
    print(f"After filtering: {len(data):,} cells")

# Verify required columns
required_cols = [CLONE_COLUMN] + SPATIAL_COLS + GROUP_COLS
missing_cols = [col for col in required_cols if col not in data.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

# Remove cells without clone assignments
data = data[data[CLONE_COLUMN].notna()].copy()
print(f"Cells with clone assignments: {len(data):,}")

# Summary statistics
n_clones = data[CLONE_COLUMN].nunique()
n_regions = len(data.groupby(GROUP_COLS))

print(f"\nData summary:")
print(f" Total cells: {len(data):,}")
print(f" Number of clones: {n_clones}")
print(f" Number of spatial regions: {n_regions}")
print(f" Cells per clone: {data.groupby(CLONE_COLUMN).size().describe()}")

# Display first few rows
display(data[[CLONE_COLUMN] + SPATIAL_COLS + GROUP_COLS].head())

## Step 2: Compute Spatial Metrics

Calculate comprehensive spatial metrics for each clone in each ROI:

- **Cell count & proportion**: Number and fraction of cells in each clone
- **Centroid location**: Spatial center of each clone
- **Dispersion**: How spread out cells are from the centroid
- **Spatial extent**: Overall size/span of the clone
- **Density**: Cells per unit area
- **Nearest neighbor**: Average spacing between clone members

In [ ]:
# =============================================================================
# STEP 2: COMPUTE SPATIAL METRICS
# =============================================================================
print("Step 2: Computing spatial metrics for each clone...")

spatial_metrics = compute_clone_spatial_metrics(
    df=data,
    clone_col=CLONE_COLUMN,
    group_cols=GROUP_COLS,
    spatial_cols=SPATIAL_COLS
)

# Save results
output_file = OUTPUT_DIR / f'{CLONE_COLUMN}_spatial_metrics.csv'
spatial_metrics.to_csv(output_file, index=False)
print(f"Saved spatial metrics: {output_file}")

print(f"\nComputed metrics for {len(spatial_metrics)} clone-ROI combinations")
print(f"Clones analyzed: {spatial_metrics['clone'].nunique()}")
print(f"ROIs analyzed: {len(spatial_metrics.groupby(GROUP_COLS))}")

# Display summary
display(spatial_metrics.head(10))

# Summary statistics
print("\nSpatial metrics summary:")
print(spatial_metrics[['cell_count', 'density', 'mean_dist_from_centroid', 
                       'mean_nn_distance', 'spatial_extent']].describe())

## Step 3: Test Spatial Randomness

Test whether each clone exhibits:
- **Random** distribution (Complete Spatial Randomness, CSR)
- **Clustered** distribution (cells closer together than expected by chance)
- **Dispersed** distribution (cells more spread out than expected)

Uses Monte Carlo simulation with 1000 iterations to generate null distribution under CSR.

In [ ]:
# =============================================================================
# STEP 3: TEST SPATIAL RANDOMNESS
# =============================================================================
print("Step 3: Testing spatial randomness (Monte Carlo simulation)...")
print(f"Running {N_SIMULATIONS} simulations per clone-ROI combination...")
print("This may take a few minutes...")

randomness_test = test_spatial_randomness(
    df=data,
    clone_col=CLONE_COLUMN,
    group_cols=GROUP_COLS,
    spatial_cols=SPATIAL_COLS,
    n_simulations=N_SIMULATIONS,
    random_seed=RANDOM_SEED
)

# Save results
output_file = OUTPUT_DIR / f'{CLONE_COLUMN}_spatial_randomness_test.csv'
randomness_test.to_csv(output_file, index=False)
print(f"Saved randomness test results: {output_file}")

# Summary of spatial patterns
pattern_counts = randomness_test['spatial_pattern'].value_counts()
print(f"\nSpatial pattern distribution:")
for pattern, count in pattern_counts.items():
    print(f" {pattern}: {count} ({count/len(randomness_test)*100:.1f}%)")

# Show significant patterns
significant = randomness_test[randomness_test['p_value'] < 0.05]
print(f"\nSignificant spatial patterns (p < 0.05): {len(significant)} / {len(randomness_test)}")

display(randomness_test.sort_values('p_value').head(10))

## Step 4: Analyze Clone-Clone Interactions

Compute spatial interactions between different clones:
- Count neighbors of each clone type within specified radius
- Identify which clones are spatial neighbors
- Generate interaction matrices for each ROI

In [ ]:
# =============================================================================
# STEP 4: CLONE-CLONE INTERACTIONS
# =============================================================================
print("Step 4: Computing clone-clone spatial interactions...")
print(f"Interaction radius: {INTERACTION_RADIUS} units")

clone_interactions = compute_clone_interactions(
    df=data,
    clone_col=CLONE_COLUMN,
    group_cols=GROUP_COLS,
    spatial_cols=SPATIAL_COLS,
    interaction_radius=INTERACTION_RADIUS
)

# Save results
output_file = OUTPUT_DIR / f'{CLONE_COLUMN}_clone_interactions.csv'
clone_interactions.to_csv(output_file, index=False)
print(f"Saved interaction results: {output_file}")

# Compute normalized interaction frequencies
clone_interactions['interaction_frequency'] = (
    clone_interactions['interaction_count'] / clone_interactions['cells_clone_a']
)

# Aggregate across all ROIs for overall patterns
overall_interactions = clone_interactions.groupby(['clone_a', 'clone_b']).agg({
    'interaction_count': 'sum',
    'cells_clone_a': 'sum',
    'cells_clone_b': 'sum'
}).reset_index()

overall_interactions['interaction_frequency'] = (
    overall_interactions['interaction_count'] / overall_interactions['cells_clone_a']
)

print(f"\nComputed interactions for {clone_interactions['clone_a'].nunique()} clones")
print(f"Total clone pairs analyzed: {len(overall_interactions)}")

# Show top interactions (excluding self-interactions)
top_interactions = overall_interactions[
    overall_interactions['clone_a'] != overall_interactions['clone_b']
].sort_values('interaction_count', ascending=False).head(10)

print("\nTop 10 clone-clone interactions:")
display(top_interactions)

## Step 5: Boundary vs Core Analysis

Test whether clones are enriched at the tumor periphery (boundary) or interior (core):

1. Compute distance from each cell to ROI boundary
2. Classify cells as boundary vs core
3. Test for clone-specific enrichment patterns

In [ ]:
# =============================================================================
# STEP 5: BOUNDARY VS CORE ANALYSIS
# =============================================================================
print("Step 5: Computing distance from boundary...")

# Compute boundary distances
data_with_boundary = compute_distance_from_boundary(
    df=data,
    group_cols=GROUP_COLS,
    spatial_cols=SPATIAL_COLS
)

print(f"Boundary threshold: {BOUNDARY_THRESHOLD} units")
print(f"Testing clone enrichment at boundary vs core...")

# Test enrichment
boundary_test = test_clone_boundary_enrichment(
    df=data_with_boundary,
    clone_col=CLONE_COLUMN,
    boundary_threshold=BOUNDARY_THRESHOLD
)

# Save results
output_file = OUTPUT_DIR / f'{CLONE_COLUMN}_boundary_enrichment.csv'
boundary_test.to_csv(output_file, index=False)
print(f"Saved boundary enrichment results: {output_file}")

# Summary
enrichment_counts = boundary_test['enrichment'].value_counts()
print(f"\nBoundary enrichment patterns:")
for pattern, count in enrichment_counts.items():
    print(f" {pattern}: {count}")

# Show significant enrichments
significant_enrichment = boundary_test[boundary_test['p_value'] < 0.05]
print(f"\nSignificant enrichments (p < 0.05): {len(significant_enrichment)} / {len(boundary_test)}")

display(boundary_test.sort_values('p_value').head(10))

## Step 6: Visualization - Spatial Distribution Maps

Generate spatial distribution maps showing clone locations within each ROI.

In [ ]:
# =============================================================================
# STEP 6: SPATIAL DISTRIBUTION PLOTS
# =============================================================================
print("Step 6: Generating spatial distribution plots...")
print(f"Saving to: {PLOTS_DIR}")

plot_clone_spatial_distribution(
    df=data,
    clone_col=CLONE_COLUMN,
    group_cols=GROUP_COLS,
    spatial_cols=SPATIAL_COLS,
    output_dir=PLOTS_DIR / 'spatial_maps',
    max_plots=20,  # Adjust based on how many ROIs you want to plot
    figsize=(10, 10)
)

print("Spatial distribution plots complete.")

## Step 7: Summary Visualizations

Create summary plots of spatial metrics across all clones.

In [ ]:
# =============================================================================
# STEP 7: SUMMARY VISUALIZATIONS
# =============================================================================
print("Step 7: Creating summary visualizations...")

# Plot spatial metrics summary
plot_clone_metrics_summary(
    spatial_metrics_df=spatial_metrics,
    output_dir=PLOTS_DIR
)

print(f"Summary plots saved to: {PLOTS_DIR}")

## Step 8: Additional Visualizations

Create custom plots for publication-quality figures.

In [ ]:
# =============================================================================
# STEP 8: CUSTOM VISUALIZATIONS
# =============================================================================
print("Step 8: Creating custom visualizations...")

# 1. Spatial pattern distribution
fig, ax = plt.subplots(figsize=(8, 6))
pattern_counts = randomness_test['spatial_pattern'].value_counts()
colors = {'Random': 'gray', 'Clustered': 'red', 'Dispersed': 'blue'}
bars = ax.bar(pattern_counts.index, pattern_counts.values, 
             color=[colors.get(p, 'gray') for p in pattern_counts.index])
ax.set_xlabel('Spatial Pattern')
ax.set_ylabel('Number of Clones')
ax.set_title(f'Spatial Pattern Distribution Across Clones\n(p < 0.05 threshold)')
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'spatial_pattern_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

# 2. Clone size vs spatial dispersion
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(spatial_metrics['cell_count'], 
          spatial_metrics['mean_dist_from_centroid'],
          alpha=0.5, s=50)
ax.set_xlabel('Clone Size (cells)')
ax.set_ylabel('Mean Distance from Centroid')
ax.set_title('Clone Size vs Spatial Dispersion')
ax.set_xscale('log')
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'size_vs_dispersion.png', dpi=300, bbox_inches='tight')
plt.show()

# 3. Boundary enrichment heatmap
if len(boundary_test) > 0:
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Sort by proportion at boundary
    boundary_sorted = boundary_test.sort_values('proportion_boundary', ascending=False).head(20)
    
    colors_map = {'Boundary-enriched': 'red', 'Core-enriched': 'blue', 
                 'No enrichment': 'gray', 'Insufficient data': 'lightgray'}
    colors = [colors_map.get(e, 'gray') for e in boundary_sorted['enrichment']]
    
    ax.barh(range(len(boundary_sorted)), boundary_sorted['proportion_boundary'], color=colors)
    ax.set_yticks(range(len(boundary_sorted)))
    ax.set_yticklabels([f"Clone {c}" for c in boundary_sorted['clone']])
    ax.set_xlabel('Proportion of Cells at Boundary')
    ax.set_title('Clone Enrichment at Tumor Boundary vs Core\n(Top 20 clones by boundary proportion)')
    ax.axvline(0.5, color='black', linestyle='--', linewidth=0.5)
    
    # Add legend
    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor=color, label=label) 
                      for label, color in colors_map.items()]
    ax.legend(handles=legend_elements, loc='lower right')
    
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / 'boundary_enrichment.png', dpi=300, bbox_inches='tight')
    plt.show()

# 4. Clone interaction heatmap (for a single representative ROI)
if len(overall_interactions) > 0:
    # Create interaction matrix
    clones = sorted(overall_interactions['clone_a'].unique())
    interaction_matrix = pd.DataFrame(0, index=clones, columns=clones)
    
    for _, row in overall_interactions.iterrows():
        interaction_matrix.loc[row['clone_a'], row['clone_b']] = row['interaction_frequency']
    
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(interaction_matrix, annot=False, cmap='YlOrRd', ax=ax, 
               cbar_kws={'label': 'Interaction Frequency'})
    ax.set_xlabel('Clone B')
    ax.set_ylabel('Clone A')
    ax.set_title(f'Clone-Clone Spatial Interactions\n(Radius: {INTERACTION_RADIUS} units)')
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / 'clone_interactions_heatmap.png', dpi=300, bbox_inches='tight')
    plt.show()

print("Custom visualizations complete.")

## Summary

Spatial analysis workflow completed successfully.

### Generated Outputs

**CSV Files:**
- `*_spatial_metrics.csv` - Comprehensive spatial metrics per clone
- `*_spatial_randomness_test.csv` - Statistical tests for spatial patterns
- `*_clone_interactions.csv` - Clone-clone spatial interactions
- `*_boundary_enrichment.csv` - Boundary vs core enrichment analysis

**Visualizations:**
- Individual spatial distribution maps per ROI
- Summary plots of spatial metrics
- Custom publication-ready figures

### Key Findings to Report

1. **Spatial Patterns**: Are clones spatially clustered, dispersed, or random?
2. **Clone Diversity**: How many distinct clones per patient/ROI?
3. **Spatial Organization**: Do certain clones preferentially localize together?
4. **Boundary Enrichment**: Are specific clones enriched at tumor periphery?
5. **Clone Interactions**: Which clones are spatial neighbors?

### Next Steps

- Correlate spatial patterns with clinical outcomes
- Analyze clone-microenvironment interactions
- Compare spatial patterns across patient groups
- Integrate with survival analysis